In [139]:
import numpy as np
import scipy

In [ ]:
def DMD_one_basis(Q, Gyx, Gx, uw=True):
    """
    DMD for HWR-sDMD in one basis.
    
    Q = orthogonal basis for S
    Gyx = cross product of S_y S_x^T
    Gx = cross product of S_x S_x^T
    uw = if you want Q(:, :rx), W to be returned instead of DMD modes Z=Q(:, :rx)W
    """
    rx = Gx.shape[0]
    B = scipy.linalg.solve(Gx.T, Gyx[:rx,:].T, assume_a='her')
    L, W = np.linalg.eig(B.T)

    r = Q.shape[1]
    if(r>rx):
        res = np.linalg.norm(scipy.linalg.solve(Gx.T, Gyx.T, assume_a='her').T@W - np.multiply(np.vstack((W, np.zeros(W.shape[1]))), L), axis=0)
    else:
        res = np.linalg.norm(B.T@W - np.multiply(W, L), axis=0)

    if uw:
        return Q[:, :rx], W, L, res
    else:
        return Q[:,:rx]@W, L, res

In [ ]:
def DMD_one_basisTQ(Q, Gyq, T, uw=True, tri='L'):
    """
    DMD for TQ-sDMD in one basis.

    Q = orthogonal basis for S
    Gyq = cross product of S_y \widetilde Q
    T = triangular matrix from TQ decomposition of S_x
    uw = if you want Q(:, :rx), W to be returned instead of DMD modes Z=Q(:, :rx)W
    tri = character 'L' or 'U' indicating if T is lower or upper triangular
    """
    rx = T.shape[0]
    if(tri=='L'):
        B = scipy.linalg.solve_triangular(T.T, Gyq[:rx,:].T)
    elif(tri == 'U'):
        B = scipy.linalg.solve_triangular(T.T, Gyq[:rx,:].T, lower=True)
    else:
        B = Gyq[:rx,:]@np.linalg.pinv(T)
    L, W = np.linalg.eig(B.T)

    #rez PROVJERI
    r = Q.shape[1]
    if(r>rx):
        res = np.linalg.norm(scipy.linalg.solve_triangular(T.T, Gyq.T, lower=True).T@W - np.multiply(np.vstack((W, np.zeros(W.shape[1]))), L), axis=0)
    else:
        res = np.linalg.norm(B.T@W - np.multiply(W, L), axis=0)

    if uw:
        return Q[:, :rx], W, L, res
    else:
        return Q[:,:rx]@W, L, res

In [ ]:
def DMD_compress(Qx, Qxy, Gyx, Gx, uw=False):
    """
    DMD algorithm for already updated Gyx, Gx from HWR-sDMD in two bases.
    """
    L, W = np.linalg.eig(Qxy@(scipy.linalg.solve(Gx.T, Gyx.T, assume_a='pos').T))
    if(uw):
        return Qx, W, L
    return Qx@W, L

In [ ]:
def DMD_TQ_two(Qx, Qxy, Gyq, T, uw=False, tri='L'):
    """
    DMD algorithm for TQ-sDMD in two bases.
    """
    if(tri=='L'):
        L, W = np.linalg.eig(Qxy@(scipy.linalg.solve(T.T, Gyq.T, assume_a='upper triangular').T))
    else:
        L, W = np.linalg.eig(Qxy@(scipy.linalg.solve(T.T, Gyq.T, assume_a='lower triangular').T))
    if(uw):
        return Qx, W, L
    return Qx@W, L

In [ ]:
def DMD_compress_original(Qx, Qxy, Gyx, Gx, uw=False):
    """
    DMD algorithm for already updated Gyx, Gx from HWR-sDMD in two bases or Gyq, T from TQ-sDMD (doesn't use special structure).
    """
    L, W = np.linalg.eig(Qxy@Gyx@np.linalg.pinv(Gx))
    if(uw):
        return Qx, W, L
    return Qx@W, L

In [ ]:
def GS_reorth(Q, x, tol1=1e-10, tol2=1e-1):
    """
    Gram-Schmidt reorthogonalization algorithm.

    Q = orthogonal matrix
    x = new vector
    tol1 = tolerance to check if x provides a new direction
    tol2 = tolerance to check if reorthogonalization is needed (tol2 >= tol1)
    """
    xtilde = Q.T@(x.reshape(-1,1))
    ex = x.reshape(-1,1)-Q@(xtilde)
    ex_norm = np.linalg.norm(ex)
    x_norm = np.linalg.norm(x)
    if(ex_norm/x_norm > tol1):
        if(ex_norm/x_norm < tol2):
            #print("reort")
            xtilde += Q.T@ex
            ex = ex-Q@(Q.T@ex)
        gamma=np.linalg.norm(ex)
        q=ex/np.linalg.norm(ex)
    else:
        gamma=0
        q=None
    
    return q, xtilde, gamma


In [ ]:
def Addxy_compress(Qx, Qy, Qxy, Gyx, Gx, Gy, x, y, r0=0, tol1=1e-6, tol2=1e-1, tol3 = 1e-8):    
    """
    HWR-sDMD in two bases with optional compression to maximum rank r0.

    Qx = orthogonal basis for X
    Qy = orthogonal basis for Y
    Qxy = Qx@Qy.T
    Gyx = cross product of \widetilde Y and \widetilde X^T
    Gx = cross product of \widetilde X and \widetilde X^T
    Gy = cross product of \widetilde Y and \widetilde Y^T 
    x = first element of new snapshot pair
    y = second element of new snapshot pair
    r0 = maximum rank. If 0, no maximum rank is set.
    tol1 = tolerance for GS orthogonalization (tol1)
    tol2 = tolerance for GS reorthogonalization (tol2)
    tol3 = when r0 != 0, tolerance for discarding small eigenvalues
    """
    rx=Qx.shape[1]; ry=Qy.shape[1]; m=x.shape[0]

    #GS procedure
    q1, xtilde, gammax = GS_reorth(Qx, x, tol1, tol2)
    q2, ytilde, gammay = GS_reorth(Qy, y, tol1, tol2)

    # update Gyx and Gx
    Gyx += ytilde * xtilde.T 
    Gx += xtilde * xtilde.T
    Gy += ytilde * ytilde.T

    
    #expansion of basis (if necessary)
    if gammax > 0:
        if gammay > 0:
            Qxy = np.bmat([[Qxy, Qx.T@q2], [q1.T@Qy, q1.T@q2]])
            Qx = np.bmat([Qx, q1.reshape(-1,1)])
            Qy = np.bmat([Qy, q2.reshape(-1,1)])
            Gyx = np.bmat([[Gyx, ytilde*gammax], [gammay*xtilde.T, np.array([[gammay*gammax]])]])
            Gy = np.bmat([[Gy, ytilde*gammay], [gammay*ytilde.T, np.array([[gammay**2]])]])
            Gx = np.bmat([[Gx, xtilde*gammax], [gammax*xtilde.T, np.array([[gammax**2]])]]) 
            ry+=1
        else:
            Qxy = np.bmat([[Qxy], [q1.T@Qy]])
            Qx = np.bmat([Qx, q1.reshape(-1,1)])
            Gyx = np.bmat([Gyx, ytilde*gammax])
            Gx = np.bmat([[Gx, xtilde*gammax], [gammax*xtilde.T, np.array([[gammax**2]])]]) 
        rx+=1
    elif gammay > 0:
        Qxy = np.bmat([Qxy, Qx.T@q2])
        Qy = np.bmat([Qy, q2.reshape(-1,1)])
        Gyx = np.bmat([[Gyx],[gammay*xtilde.T]])
        Gy = np.bmat([[Gy, ytilde*gammay], [gammay*ytilde.T, np.array([[gammay**2]])]])
        ry += 1
    
    #compression
    if r0:
        if rx == r0:
            evals, evecs = np.linalg.eig(Gx)
            idx = np.argsort(np.abs(evals)) #in case the property of symmetry was lost
            idx = idx[-1:-1-r0:-1]   # sorted from largest to smallest
            first=idx[0]
            rho=r0
            for i in range(1, r0):
                if(evals[idx[i]]<=evals[first]*tol3):
                    rho = i
                    break
            if(rho<r0):
                idx = idx[:rho]
                qx = np.asmatrix(evecs[:, idx])
                Qx = Qx * qx
                Gyx = Gyx * qx
                Qxy = qx.T * Qxy
                Gx = np.asmatrix(np.diag(evals[idx]))
            else:
                print("Gx can't be compressed!")
        if ry > r0:
            evals, evecs = np.linalg.eig(Gy)
            idx = np.argsort(evals)
            idx = idx[-1:-1-r0:-1]   # indices of largest r0 eigenvalues
            first=idx[0]
            rho = ry
            for i in range(1, r0):
                if(evals[idx[i]]<=evals[first]*tol3):
                    rho = i
                    break
            if(rho < r0):
                idx = idx[:rho]
                qy = np.asmatrix(evecs[:, idx])
                Qy = Qy * qy
                Gyx = qy.T * Gyx
                Qxy = Qxy * qy 
                Gy = np.asmatrix(np.diag(evals[idx]))
            else:
                print("Gy can't be compressed")


    return Qx, Qy, Qxy, Gyx, Gx, Gy


In [ ]:
# GIVENS FUNCTIONS
def givens_c_s(vector, i, j):
    xi=vector[i]
    xj =vector[j]
    denom = np.sqrt(xi**2 + xj**2)
    return xi/denom, -xj/denom

def givens_c_s_elements(xi, xj):
    denom = np.sqrt(xi**2 + xj**2)
    return xi/denom, -xj/denom

def apply_givens_c_s(AA, c, s, i, j, transp = False):
    G = np.array([[c, -s], [s, c]])
    if(transp):
        AA[:, [i,j]] = AA[:,[i,j]]@(G.T)
    else:
        AA[[i,j],:] = G @ AA[[i,j],:] 
    return AA

def apply_givens_full(AA, i, j, transp=False, cs=False): 
    """
    Directly changes AA. If not wanted send np.copy(A) instead of A.
    """
    xi=AA[i,i]
    if(transp):
        xj = AA[i,j]
    else:
        xj =AA[j,i]
    denom = np.sqrt(xi**2 + xj**2)
    c=xi/denom
    s=-xj/denom
    G = np.array([[c, -s], [s, c]])
    if(transp):
        AA[:, [i,j]] = AA[:,[i,j]]@(G.T)
    else:
        AA[[i,j],:] = G @ AA[[i,j],:] 
    if(cs):
        return AA, c, s
    return AA

In [ ]:
def updateYQT(YQ, T, xtilde, ytilde, gammax, gammay, tri='L'): #staro ime YQL_update
    """ 
    function for updates of Gyq (=YQ) and T. For TQ-sDMD in both one and two bases.

    YQ = cross product of Y and Q (also denoted as Gyq)
    T = upper triangular matrix from TQ decomposition of X
    xtilde = first element of snapshot pair in lower dimension space
    ytilde = second element of snapshot pair in lower dimension space
    gammax = indicating if snapshot x expands basis (typically result of GS_reorth)
    gammay = indicating if snapshot y expands basis (typically result of GS_reorth)
    tri = character 'L' or 'U' indicating if T is upper or lower triangular
    """
    ry=YQ.shape[0]
    rx=YQ.shape[1]
    if(gammax!=0):
        T = np.vstack((T, np.zeros((1, rx))))
    
    if(gammay!=0):
        YQ = np.vstack((YQ, np.zeros((1, rx))))

    YQ = np.hstack((YQ, ytilde))
    T = np.hstack((T, xtilde))

    #Application of Givens rotation (if necessary)
    if(tri=='L'):
        for i in range(rx):
            c, s = givens_c_s_elements(T[i,i],  T[i,rx])
            apply_givens_c_s(T, c, s, i, rx, True)
            apply_givens_c_s(YQ, c, s, i, rx, True)
    elif (gammax==0):
        for i in range(rx-1, -1, -1):
            c, s = givens_c_s_elements(T[i,i],T[i,rx])
            apply_givens_c_s(T, c, s, i, rx, True)
            apply_givens_c_s(YQ, c, s, i, rx, True)
        
    if(gammax==0):
        YQ = YQ[:, :-1]
        T = T[:,:-1]

    return YQ, T


In [ ]:
def AddxyTQ(Qx, Qy, Gyq, T, x, y, tol1=1e-6, tol2 = 1e-1, tri='L'): 
    """
    Updates for TQ-sDMD in two bases and no maximal rank.

    Qx = orthogonal basis for X
    Qy = orthogonal basis for Y
    Gyq = \widetilde Y @ \widetilde X^T
    T = triangular matrix from TQ decomposition of \widetilde X
    x = first element of new snapshot pair
    y = second element of new snapshot pair
    tol1 = tol1 for GS_reorth
    tol2 = tol2 for GS_reorth
    tri = character 'L' or 'U' indicating if T is upper or lower triangular
    """
    rx=T.shape[0]
    ry=Gyq.shape[0]
    qx,gx,gammax = GS_reorth(Qx, x, tol1, tol2)
    qy,gy,gammay = GS_reorth(Qy, y, tol1, tol2)
    if(gammax):
        xtilde = np.vstack((gx,gammax))
        Qx = np.hstack((Qx, qx.reshape(-1,1)))
    else:
        xtilde=gx
    if(gammay):
        ytilde = np.vstack((gy,gammay))
        Qy = np.hstack((Qy, qy.reshape(-1,1)))
    else:
        ytilde = gy

    Gyq, T = updateYQT(Gyq, T, xtilde, ytilde, gammax, gammay, tri)
    
    return Qx, Qy, Gyq, T

In [ ]:
def AddxyTQr(Qx, Qy, tQyx, Tx, Ty, x, y, r0=0, tol1=1e-6, tol2=1e-1, tol3=1e-8, trix='L', triy='L'):
    """
    Updates for TQ-sDMD in two bases with maximal rank.

    Qx = orthogonal basis for X
    Qy = orthogonal basis for Y
    tQyx = cross product \widetilde Q_y^T \widetilde Q_x (not Qx@Qy.T !)
    Tx = triangular matrix from TQ decomposition of \widetilde X
    Ty = triangular matrix from TQ decomposition of \widetilde Y
    x = first element of new snapshot pair
    y = second element of new snapshot pair
    r0 = maximal rank. If r0 = 0 (default), ignored.
    tol1 = tol1 for GS_reorth
    tol2 = tol2 for GS_reorth
    tol3 = tolerance for reducing rank when maximal rank is achieved.
    trix = character 'L' or 'U' indicating if Tx is upper or lower triangular
    triy = character 'L' or 'U' indicating if Ty is upper or lower triangular
    """
    (ry, rx) = tQyx.shape
    qx,gx,gammax = GS_reorth(Qx, x, tol1, tol2)
    qy,gy,gammay = GS_reorth(Qy, y, tol1, tol2)

    tQyx = np.hstack((tQyx, np.zeros((ry,1))))
    tQyx = np.vstack((tQyx, np.eye(1, rx+1, rx)))
    Tx = np.hstack((Tx, gx))
    Ty = np.hstack((Ty, gy))   

    if(gammax>0):
        temp = np.hstack((np.zeros(1,rx), [[gammax]]))
        Tx = np.vstack((Tx, temp))
        Qx = np.hstack((Qx, qx))
        rx+=1
    if(gammay > 0):
        temp = np.hstack((np.zeros(1,ry), [[gammay]]))
        Ty = np.vstack((Ty, temp))
        Qy = np.hstack((Qy, qy))
        ry+=1

    #Application of Givens rotation (if necessary) for X
    if(trix=='L'):
        for i in range(rx):
            c, s = givens_c_s_elements(Tx[i,i],  Tx[i,rx])
            apply_givens_c_s(Tx, c, s, i, rx, True)
            apply_givens_c_s(tQyx, c, s, i, rx, True)
    elif (gammax==0):
        for i in range(rx-1, -1, -1):
            c, s = givens_c_s_elements(Tx[i,i],Tx[i,rx])
            apply_givens_c_s(Tx, c, s, i, rx, True)
            apply_givens_c_s(tQyx, c, s, i, rx, True)

    #Application of Givens rotation (if necessary) for Y
    if(triy=='L'):
        for i in range(ry):
            c, s = givens_c_s_elements(Ty[i,i],  Ty[i,ry])
            apply_givens_c_s(Ty, c, s, i, ry, True)
            apply_givens_c_s(tQyx, c, s, i, ry)
    elif (gammay==0):
        for i in range(ry-1, -1, -1):
            c, s = givens_c_s_elements(Ty[i,i],Ty[i,ry])
            apply_givens_c_s(Ty, c, s, i, ry, True)
            apply_givens_c_s(tQyx, c, s, i, ry)

    #cutting if Qx or Qy wasn't expanded
    if(gammax == 0):
        Tx = Tx[:, :-1]
        tQyx = tQyx[:, :-1]
    if(gammay == 0):
        Ty = Ty[:, :-1]
        tQyx = tQyx[:-1,:]
    
    (ry, rx) = tQyx.shape

    #maximum rank
    if r0:
        if rx == r0:
            U, S, V = np.linalg.svd(Tx, full_matrices=False)
            start = S[0]
            rho=rx
            for i in range(1,rx):
                if(S[i]< tol3*start):
                    rho=i
                    break
            if(rho<rx):
                Qx = Qx@(U[:, 1:rho])
                tQyx = tQyx @ np.conj(V[:rho,:].T)
                Tx = np.diag(S[:rho])
            else:
                print("Tx can't be compressed")
        if ry == r0:
            U, S, V = np.linalg.svd(Ty, full_matrices=False)
            start = S[0]
            rho=ry
            for i in range(1,ry):
                if(S[i]< tol3*start):
                    rho=i
                    break
            if(rho<ry):
                Qy = Qy@(U[:, 1:rho])
                tQyx = V[:rho,:] @ tQyx
                Tx = np.diag(S[:rho])
            else:
                print("Ty can't be compressed")
    

In [ ]:
def Addx_one_basis(Q, Gyx, Gx, x, xt_prev, tol1=1e-6, tol2=1e-1, tol3=1e-8, r0=0): 
    """
    updating HWR-sDMD in one basis.

    Q = orthogonal basis for S
    Gyx = cross product of Y and X^T
    Gx = cross product of X and X^T
    x = new snapshot 
    xt_prev = \widetilde x from previously received snapshot (in lower dimension)
    tol1 = tol1 for GS_reorth
    tol2 = tol2 for GS_reorth
    tol3 = tolerance for maximal rank reduction
    r0 = maximal rank (default 0 - ignored)
    """

    r = Q.shape[1]
    rx = Gx.shape[0]
    
    if(r>rx): ### or add parameter increase!
        Gyx = np.hstack((Gyx, np.zeros((rx+1,1)))) 
        Gx = np.bmat([[Gx, np.zeros((rx,1))], [np.zeros((1, rx+1))]])
        rx+=1

    Gx += xt_prev@xt_prev.T

    if r0:
        if (r==r0):
            evals, evecs = np.linalg.eig(Gx)
            idx = np.argsort(np.abs(evals)) #in case the property of symmetry was lost we put abs
            idx = idx[-1:-1-r0:-1]   # sorted from largest to smallest
            first=idx[0]
            rho=r0
            for i in range(1, r0):
                if(evals[idx[i]]<=evals[first]*tol3):
                    rho = i
                    break
            if(rho < r0):
                idx = idx[:rho]
                qx = np.asmatrix(evecs[:, idx])
                Q = Q@qx
                Gx = np.diag(evals[idx])
                xt_prev = qx.T@xt_prev
            else:
                print("Q can't be compressed")

    q, xt, gamma = GS_reorth(Q, x, tol1, tol2)

    if(gamma>0):
        Q = np.hstack((Q, q))
        Gyx = np.vstack((Gyx, np.zeros((1, rx))))
        xt = np.vstack((xt, gamma))
    
    Gyx += xt@xt_prev.T

    return Q, Gyx, Gx, xt

In [ ]:
def AddxTQ_one_basis(Q, Gyq, T, x, xt_prev, tol1=1e-6, tol2=1e-1, tri='L'):
    """
    updating TQ-sDMD in one basis, no maximal rank.

    Q = orthogonal basis for S
    Gyq = cross product of Y and \widetilde Q
    T = triangular matrix from TQ decomposition of \widetilde X
    x = new snapshot 
    xt_prev = \widetilde x from previously received snapshot (in lower dimension)
    tol1 = tol1 for GS_reorth
    tol2 = tol2 for GS_reorth
    tri = character indicating if T is upper ('U') or lower ('L') triangular
    """
    rx=T.shape[0]
    r=Q.shape[1]
    
    q,xtilde,gamma = GS_reorth(Q, x, tol1, tol2)

    if(r>rx): ### OR increase 
        gammax = 1
    else:
        gammax=0
        
    if(gamma):
        Q = np.hstack((Q, q.reshape(-1,1)))
        xtilde = np.vstack((xtilde.reshape(-1,1), gamma))
        r+=1

    Gyq, T = updateYQT(Gyq, T, xt_prev, xtilde, gammax, gamma, tri)
   
    return Q, Gyq, T, xtilde

In [ ]:
def AddxTQr_one_basis(Q, Gyq, T, x, xt_prev, tol1=1e-6, tol2=1e-1, tol3=1e-8, r0=0, tri='L'):
    """
    updating TQ-sDMD in one basis with maximal rank.

    Q = orthogonal basis for S
    Gyq = cross product of Y and \widetilde Q
    T = triangular matrix from TQ decomposition of \widetilde X
    x = new snapshot 
    xt_prev = \widetilde x from previously received snapshot (in lower dimension)
    tol1 = tol1 for GS_reorth
    tol2 = tol2 for GS_reorth
    tol3 = tolerance for reduction when maximal rank is achieved
    r0 = maximal rank
    tri = character indicating if T is upper ('U') or lower ('L') triangular
    """
    rx=T.shape[0]
    r=Q.shape[1]

    if(r>rx): ### ILI increase 
        gammax = 1
    else:
        gammax=0

    if r0:
        if(r==r0 and r>rx): #only entering if it was expanded to max in the previous set
            T = np.vstack((T, np.zeros((1, rx))))
            T = np.hstack((T, xt_prev)) 
            U, S, V = np.linalg.svd(T)
            rho = r0
            start=S[0]
            for i in range(1,r0):
                if(S[i] < tol3*start):
                    rho=i
                    break
            if(rho < r0):
                Q = Q@(U[:, :rho])
                Gyq = np.conj(U[:, :rho]).T@Gyq@V[:rho,:]
                T = np.diag(S[:rho])
                xt_prev = T[:,-1].reshape(-1,1)
                T = T[:, :-1]
                gammax=0
                tri='U' #T is diagonal, can be seen as upper or lower triangular, but potentially cheaper further operations if upper t.
            else:
                print("Q can't be compressed")
    
    q,xtilde,gamma = GS_reorth(Q, x, tol1, tol2)
        
    if(gamma):
        Q = np.hstack((Q, q.reshape(-1,1)))
        xtilde = np.vstack((xtilde.reshape(-1,1), gamma))
        r+=1

    Gyq, T = updateYQT(Gyq, T, xt_prev, xtilde, gammax, gamma, tri) 
   
    return Q, Gyq, T, xtilde

In [ ]:
def R_update(R, x_new, remove=True):
    """
    Updates of R for Chol-sDMD and Exp-sDMD.

    R = upper triangular matrix from Cholesky decomposition
    x_new = new snapshot of same length as either dimension of R
    remove = if, after moving x_new onto R, this new column should be removed.
    """
    Rnew = np.vstack((R, x_new.reshape(1,-1)))
    rx = R.shape[0] #old length
    for i in range(rx):
        c, s = givens_c_s_elements(Rnew[i,i],  Rnew[rx,i])
        Rnew[[i,rx],:] = apply_givens_c_s(Rnew[[i,rx],:], c, s, 0, 1) 
    if remove:
        Rnew = Rnew[:-1,:]

    return Rnew

In [ ]:
def DMD_alpha_for_reconstruction(X, Z, indices, L):
    """
    helper function for obtaining alpha's from reconstruction problem.
    
    X = snapshot matrix (possibly in low-rank representation)
    Z = dmd modes (possibly in low-rank representation)
    indices = indices which we want to work with OR 'all' indicating we want to use all indices.
    """
    if (isinstance(indices, str) and indices == 'all'): 
        indices=np.array([i for i in range (Z.shape[1])])
    m=X.shape[1]; l=indices.shape[0]
    Z = Z[:,indices]
    Q, R = np.linalg.qr(Z) # Q is N x l, R is lxl
    pom=np.vander(L[indices], m, increasing=True)
    alpha= np.multiply(np.conj(R.T)@R, np.conj(pom @ np.conj(pom.T)))
    G = (np.conj(Q.T) @  X)[:l, :]  
    alpha = scipy.linalg.solve(alpha, np.multiply(np.conj(pom),(np.conj(R.T)@G))@np.ones((m, 1)), assume_a='pos') 
    return Z, L[indices], alpha.reshape(-1)

def DMD_reconstruction(X, Z, indices, L, times, real=True, uw=False, U=None, W=None): 
    """
    Algorithm for reconstruction based on minimization problem.
    
    X = snapshot matrix 
    Z = dmd modes (returned from some version of DMD)
    indices = indices that will be used
    L = dmd eigs (returned from some version of DMD)
    times = list of times which you want to reconstruct/predict
    real = if snapshots are real or complex. Deafault is True.
    uw = if instead of Z, U and W are given (Z = UW). Default is False.
    U = orthogonal basis of X
    W = eigs of Rayleigh-Ritz matrix
    """
    if(uw and W is not None and U is not None):
        W_l, L, alpha = DMD_alpha_for_reconstruction(np.conj(U.T)@X, W, indices, L)
        num=np.asarray(times).shape[0]
        recs = np.empty((W_l.shape[0], num), dtype=complex)
        for i in range(num):
            recs[:,i] = W_l@(L**(times[i])*alpha) 
        if real:
            return np.real(U@recs) 
        else:
            return U@recs
    else:
        Z_l, L, alpha = DMD_alpha_for_reconstruction(X, Z, indices, L)
        num=np.asarray(times).shape[0]
        recs = np.empty((Z_l.shape[0], num), dtype=complex)
        for i in range(num):
            recs[:,i] = Z_l@(L**(times[i])*alpha) 
        if real:
            return np.real(recs) 
        else:
            return recs
